In [36]:
import os
import json
import numpy as np
import plotly.graph_objects as go

In [37]:
def read_calibration_raw(input_file):
    if not os.path.exists(input_file):
        print(f"Error: {input_file} not found.")
        return

    with open(input_file, 'r') as f:
        data = json.load(f)

    cameras = data.get('cameras', [])
    return cameras

def read_calibration_panoptic(json_path):
    with open(json_path, 'r') as f:
        data = json.load(f)
    
    cameras = data.get('customized_sequence', [])
    return cameras

def invert_extrinsics(cameras):
    modified_cameras = []

    for cam in cameras:
        intr = cam.get("intrinsics", {})
        extr = cam.get("extrinsics", {})

        vm = extr.get("view_matrix", None)
        if vm is None:
            # Nothing to invert; keep as-is
            new_cam = dict(cam)
            new_cam["intrinsics"] = intr
            new_cam["extrinsics"] = extr
            modified_cameras.append(new_cam)
            continue

        vm = list(vm)
        if len(vm) < 12:
            raise ValueError(f"extrinsics.view_matrix must have at least 12 elements, got {len(vm)}")

        # Extract R, T from the 3x4 view matrix (row-major)
        R = np.array([
            [vm[0],  vm[1],  vm[2]],
            [vm[4],  vm[5],  vm[6]],
            [vm[8],  vm[9],  vm[10]],
        ], dtype=np.float64)

        T = np.array([
            [vm[3]],
            [vm[7]],
            [vm[11]],
        ], dtype=np.float64)

        # Invert: P_world = R^T * P_cam - R^T * T
        R_inv = R.T
        T_inv = -R_inv @ T

        # Write back as a 3x4 row-major matrix, preserving any trailing elements
        vm_inv_12 = [
            float(R_inv[0, 0]), float(R_inv[0, 1]), float(R_inv[0, 2]), float(T_inv[0, 0]),
            float(R_inv[1, 0]), float(R_inv[1, 1]), float(R_inv[1, 2]), float(T_inv[1, 0]),
            float(R_inv[2, 0]), float(R_inv[2, 1]), float(R_inv[2, 2]), float(T_inv[2, 0]),
        ]
        if len(vm) > 12:
            vm_inv = vm_inv_12 + vm[12:]  # keep extra data if present
        else:
            vm_inv = vm_inv_12

        new_extr = dict(extr)
        new_extr["view_matrix"] = vm_inv

        new_cam = dict(cam)
        new_cam["intrinsics"] = intr
        new_cam["extrinsics"] = new_extr

        modified_cameras.append(new_cam)

    return modified_cameras

def modify_extrinsics(cameras):
    modified_cameras = []
    for cam in cameras:
        intr = cam.get('intrinsics', {}).copy()
        extr = cam.get('extrinsics', {}).copy()
        vm = extr.get('view_matrix', None)
        v = list(vm)

        # Swap Rotation Columns 1 and 2
        v[1], v[2] = v[2], v[1]
        v[5], v[6] = v[6], v[5]
        v[9], v[10] = v[10], v[9]
        v[7], v[11] = v[11], v[7]

        # Invert the new Z axis
        v[2] = -v[2]
        v[6] = -v[6]
        v[10] = -v[10]
        v[11] = -v[11]

        new_extr = dict(extr)
        new_extr['view_matrix'] = v

        new_cam = dict(cam)
        new_cam['intrinsics'] = intr
        new_cam['extrinsics'] = new_extr
        
        modified_cameras.append(new_cam)
        
    return modified_cameras

def convert_calibration_to_panoptic(cameras, scale_to_mm=True):
    customized_sequence = []
    
    # Define scaling factor
    scale = 1000.0 if scale_to_mm else 1.0

    for cam in cameras:
        intr = cam.get('intrinsics', {})
        extr = cam.get('extrinsics', {})
        
        # 1. Extract Intrinsics (These are usually in pixels, so they stay as is)
        cm = intr.get('camera_matrix', [])
        fx, fy = cm[0], cm[4]
        cx, cy = cm[2], cm[5]

        # 2. Extract Distortion Coefficients
        dist = intr.get('distortion_coefficients', [0, 0, 0, 0, 0])
        k = [[dist[0]], [dist[1]], [dist[4]]]
        p = [[dist[2]], [dist[3]]]

        # 3. Extract Extrinsics
        vm = extr.get('view_matrix', [])
        
        R = np.array([
            [vm[0], vm[1], vm[2]],
            [vm[4], vm[5], vm[6]],
            [vm[8], vm[9], vm[10]]
        ])
        
        # Multiply T by 1000 here to convert meters to mm
        T = np.array([
            [vm[3] * scale],
            [vm[7] * scale],
            [vm[11] * scale]
        ])

        # 4. Construct the entry
        cam_entry = {
            "R": R.tolist(),
            "T": T.tolist(),
            "fx": fx,
            "fy": fy,
            "cx": cx,
            "cy": cy,
            "k": k,
            "p": p
        }
        
        customized_sequence.append(cam_entry)
    print(f"Units: {'mm' if scale_to_mm else 'meters'}")
    return customized_sequence

def save_calibration_panoptic(output_file, customized_sequence):
    output_data = {
        "customized_sequence": customized_sequence
    }

    with open(output_file, 'w') as f:
        json.dump(output_data, f, indent=4)
    
    print(f"Successfully converted {len(customized_sequence)} cameras.")

def visualize_panoptic_setup(cameras):
    fig = go.Figure()
    all_points = [] 

    # In Panoptic dataset, T is usually the world position (mm)
    centers = np.array([np.array(cam['T']).flatten() for cam in cameras])
    if len(centers) > 1:
        avg_dist = np.mean(np.linalg.norm(centers[:, None] - centers, axis=2))
        frustum_depth = avg_dist * 0.1
    else:
        frustum_depth = 300.0

    for i, cam in enumerate(cameras):
        R = np.array(cam['R'])
        T = np.array(cam['T']).flatten()
        C = T  
        
        fx, fy = cam['fx'], cam['fy']
        cx, cy = cam['cx'], cam['cy']
        w, h = cx * 2, cy * 2 

        corners_cam = np.array([
            [(0 - cx) * frustum_depth / fx, (0 - cy) * frustum_depth / fy, frustum_depth],
            [(w - cx) * frustum_depth / fx, (0 - cy) * frustum_depth / fy, frustum_depth],
            [(w - cx) * frustum_depth / fx, (h - cy) * frustum_depth / fy, frustum_depth],
            [(0 - cx) * frustum_depth / fx, (h - cy) * frustum_depth / fy, frustum_depth]
        ])
        
        corners_world = (R.T @ corners_cam.T).T + T
        all_points.extend(corners_world)
        all_points.append(C)

        for corner in corners_world:
            fig.add_trace(go.Scatter3d(
                x=[C[0], corner[0]], y=[C[1], corner[1]], z=[C[2], corner[2]],
                mode='lines', line=dict(color='blue', width=2), showlegend=False
            ))
        
        rect = np.vstack([corners_world, corners_world[0]])
        fig.add_trace(go.Scatter3d(
            x=rect[:, 0], y=rect[:, 1], z=rect[:, 2],
            mode='lines', line=dict(color='blue', width=2), showlegend=False
        ))

        fig.add_trace(go.Mesh3d(
            x=corners_world[:, 0], y=corners_world[:, 1], z=corners_world[:, 2],
            i=[0, 0], j=[1, 2], k=[2, 3], 
            opacity=0.15, color='cyan', name=f'Cam {i} FOV'
        ))

        fig.add_trace(go.Scatter3d(
            x=[C[0]], y=[C[1]], z=[C[2]],
            mode='markers+text', marker=dict(size=4, color='red'),
            text=[f"Cam {i}"], textposition="top center", name=f"Camera {i}"
        ))


    # Create the cone
    cam_positions = np.array([np.array(cam['T']).flatten() for cam in cameras])
    center_of_cameras = np.mean(cam_positions, axis=0)
    cone_height = frustum_depth * 2.0  # Make it prominent
    cone_base_radius = frustum_depth * 0.5
    theta = np.linspace(0, 2*np.pi, 20)
    bx = center_of_cameras[0] + cone_base_radius * np.cos(theta)
    by = center_of_cameras[1] + cone_base_radius * np.sin(theta)
    bz = np.full_like(theta, center_of_cameras[2])
    tip = [center_of_cameras[0], center_of_cameras[1], center_of_cameras[2] + cone_height]
    x_cone = np.append(bx, tip[0])
    y_cone = np.append(by, tip[1])
    z_cone = np.append(bz, tip[2])
    i_indices = []
    j_indices = []
    k_indices = []
    for n in range(len(theta) - 1):
        i_indices.append(n)
        j_indices.append(n + 1)
        k_indices.append(20) # Connect to tip
        
    fig.add_trace(go.Mesh3d(
        x=x_cone, y=y_cone, z=z_cone,
        i=i_indices, j=j_indices, k=k_indices,
        color='green', opacity=0.4, name='Z-Axis Pointer'
    ))

    all_points = np.array(all_points)
    center_scene = (all_points.min(axis=0) + all_points.max(axis=0)) / 2
    max_range = np.ptp(all_points, axis=0).max() / 2

    fig.update_layout(
        scene=dict(
            xaxis=dict(range=[center_scene[0]-max_range, center_scene[0]+max_range], title='X (mm)'),
            yaxis=dict(range=[center_scene[1]-max_range, center_scene[1]+max_range], title='Y (mm)'),
            zaxis=dict(range=[center_scene[2]-max_range, center_scene[2]+max_range], title='Z (mm)'),
            aspectmode='cube'
        ),
        title="Panoptic Studio Setup)",
        margin=dict(l=0, r=0, b=0, t=50)
    )

    fig.show()

In [38]:
cameras_raw = read_calibration_raw('calibration_raw.json')
cameras_raw = invert_extrinsics(cameras_raw)
cameras_raw = modify_extrinsics(cameras_raw)
customized_sequence = convert_calibration_to_panoptic(cameras_raw)
save_calibration_panoptic('calibration.json', customized_sequence)

cameras_panoptic = read_calibration_panoptic('calibration.json')
visualize_panoptic_setup(cameras_panoptic)

Units: mm
Successfully converted 5 cameras.
